<a href="https://colab.research.google.com/github/asnaraliya/ICT/blob/main/Preprocessing_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("house-prices.csv")

# Show first few rows
print("First 5 rows of dataset:")
print(df.head())


First 5 rows of dataset:
   Home   Price  SqFt  Bedrooms  Bathrooms  Offers Brick Neighborhood
0     1  114300  1790         2          2       2    No         East
1     2  114200  2030         4          2       3    No         East
2     3  114800  1740         3          2       1    No         East
3     4   94700  1980         3          2       3    No         East
4     5  119800  2130         3          3       3    No         East


In [17]:
print("Dataset Info:")
print(df.info())

print("\nSummary Statistics:")
print(df.describe())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Home          128 non-null    int64 
 1   Price         128 non-null    int64 
 2   SqFt          128 non-null    int64 
 3   Bedrooms      128 non-null    int64 
 4   Bathrooms     128 non-null    int64 
 5   Offers        128 non-null    int64 
 6   Brick         128 non-null    object
 7   Neighborhood  128 non-null    object
dtypes: int64(6), object(2)
memory usage: 8.1+ KB
None

Summary Statistics:
             Home          Price         SqFt    Bedrooms   Bathrooms  \
count  128.000000     128.000000   128.000000  128.000000  128.000000   
mean    64.500000  130427.343750  2000.937500    3.023438    2.445312   
std     37.094474   26868.770371   211.572431    0.725951    0.514492   
min      1.000000   69100.000000  1450.000000    2.000000    2.000000   
25%     32.750000  11

In [18]:
# Remove duplicate rows
df = df.drop_duplicates()

# Remove duplicate columns
df = df.loc[:, ~df.T.duplicated()]

# Verify
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate columns:", df.T.duplicated().sum())


Duplicate rows: 0
Duplicate columns: 0


In [19]:
print("Missing values before handling:")
print(df.isnull().sum())

# Numerical columns → median imputation
num_cols = df.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns → mode imputation
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing values after handling:")
print(df.isnull().sum())


Missing values before handling:
Home            0
Price           0
SqFt            0
Bedrooms        0
Bathrooms       0
Offers          0
Brick           0
Neighborhood    0
dtype: int64

Missing values after handling:
Home            0
Price           0
SqFt            0
Bedrooms        0
Bathrooms       0
Offers          0
Brick           0
Neighborhood    0
dtype: int64


/tmp/ipykernel_504/3031672025.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_504/3031672025.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

In [20]:
# Exclude target variable 'Price'
num_features = [col for col in num_cols if col != 'Price']

scaler = StandardScaler()
df[num_features] = scaler.fit_transform(df[num_features])

print("Scaled numerical features (first 5 rows):")
print(df[num_features].head())


Scaled numerical features (first 5 rows):
       Home      SqFt  Bedrooms  Bathrooms    Offers
0 -1.718572 -1.000916 -1.415327  -0.868939 -0.542769
1 -1.691507  0.137904  1.350503  -0.868939  0.396075
2 -1.664443 -1.238171 -0.032412  -0.868939 -1.481614
3 -1.637379 -0.099350 -0.032412  -0.868939  0.396075
4 -1.610315  0.612413 -0.032412   1.082362  0.396075


In [21]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("Dataset after encoding categorical variables:")
print(df.head())


Dataset after encoding categorical variables:
       Home   Price      SqFt  Bedrooms  Bathrooms    Offers  Brick_Yes  \
0 -1.718572  114300 -1.000916 -1.415327  -0.868939 -0.542769      False   
1 -1.691507  114200  0.137904  1.350503  -0.868939  0.396075      False   
2 -1.664443  114800 -1.238171 -0.032412  -0.868939 -1.481614      False   
3 -1.637379   94700 -0.099350 -0.032412  -0.868939  0.396075      False   
4 -1.610315  119800  0.612413 -0.032412   1.082362  0.396075      False   

   Neighborhood_North  Neighborhood_West  
0               False              False  
1               False              False  
2               False              False  
3               False              False  
4               False              False  


In [22]:
for col in num_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("Dataset shape after outlier removal:", df.shape)


Dataset shape after outlier removal: (63, 9)


In [23]:
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Training set shape: (50, 8)
Testing set shape: (13, 8)
